In [ ]:
import os
from pathlib import Path
import json
import random
import math
import os, math, contextlib, torch, numpy as np
from tqdm import tqdm
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.ops import FeaturePyramidNetwork

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


In [ ]:
BASE_DIR = Path(".") 
IMG_DIRS = [BASE_DIR / f"images{i}" for i in range(7)]  
TRAIN_JSON = BASE_DIR / "training.json"
TEST_JSON  = BASE_DIR / "testing.json"

IMG_SIZE = 256           
BATCH_SIZE = 8
EPOCHS = 10               
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.15
CHECKPOINT = "resnet50_fpn_binary.pth"
PRED_CSV   = "test_predictions_resnet50_fpn.csv"
PIN_MEMORY = (DEVICE == "cuda")
NUM_WORKERS = 0 if DEVICE != "cuda" else 4


In [ ]:
def find_image_path(file_name: str, img_dirs=IMG_DIRS):
    for d in img_dirs:
        p = d / file_name
        if p.exists():
            return p
    return None

def make_label(entry: dict) -> int:
    sev = entry.get("severity", 0)
    try:
        sev_val = float(sev)
    except (TypeError, ValueError):
        sev_val = 0.0
    return int(sev_val >= 1)

with open(TRAIN_JSON, "r") as f:
    tr = json.load(f)

train_items = []
for d in tr.get("images", tr):
    fn = d.get("file_name")
    p = find_image_path(fn)
    if p is not None:
        train_items.append({
            "file_name": fn,
            "path": str(p),
            "label": make_label(d),
            "width": d.get("width"),
            "height": d.get("height"),
        })
    else:
        pass

df = pd.DataFrame(train_items)
print("Train Samples:", len(df), " | Positives:", df["label"].sum(), " | Negatives:", (1 - df["label"]).sum())
df.head()


Train Samples: 9096  | Positives: 2148  | Negatives: 6948


,file_name,path,label,width,height
0,1.png,images5/1.png,1,1000,1000
1,2.png,images0/2.png,1,1000,1000
2,3.png,images3/3.png,1,1000,1000
3,4.png,images3/4.png,1,1000,1000
4,5.png,images0/5.png,1,1000,1000


In [ ]:
"""MAX_PER_CLASS = 10

def downsample_balanced(df, max_per_class=MAX_PER_CLASS, seed=SEED):
    parts = []
    for lbl, grp in df.groupby("label"):
        n = min(len(grp), max_per_class)
        parts.append(grp.sample(n=n, random_state=seed, replace=False))
    df_small = pd.concat(parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df_small

df = downsample_balanced(df, max_per_class=MAX_PER_CLASS, seed=SEED)

print("REDUCED Train Samples:", len(df),
      " | Positives:", int((df["label"]==1).sum()),
      " | Negatives:", int((df["label"]==0).sum()))
df.head()
"""

REDUCED Train Samples: 20  | Positives: 10  | Negatives: 10


,file_name,path,label,width,height
0,8105.png,images0/8105.png,0,1055,1046
1,393.png,images5/393.png,1,1000,1000
2,151.png,images4/151.png,1,1000,1000
3,4449.png,images3/4449.png,0,1000,1000
4,11064.png,images4/11064.png,0,1000,1000


In [ ]:
train_df, val_df = train_test_split(
    df, test_size=VAL_SPLIT, random_state=SEED, stratify=df["label"]
)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")

pos = (train_df["label"] == 1).sum()
neg = (train_df["label"] == 0).sum()
pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=DEVICE)
print("pos_weight =", float(pos_weight))


Train: 17  Val: 3
pos_weight = 0.8888888955116272


In [6]:
train_tfms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0, value=0),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2, rotate_limit=15, border_mode=0, value=0, p=0.7),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0, value=0),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])


/var/folders/b0/h1xx55fn0m7f9j3k4jxny4nm0000gn/T/ipykernel_2815/3330455991.py:3: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0, value=0),
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/b0/h1xx55fn0m7f9j3k4jxny4nm0000gn/T/ipykernel_2815/3330455991.py:5: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2, rotate_limit=15, border_mode=0, value=0, p=0.7),
/var/folders/b0/h1xx55fn0m7f9j3k4jxny4nm0000gn/T/ipykernel_2815/3330455991.py:13: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=0, value=

In [ ]:
class AerialWasteBinaryDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def _load_image(self, path: str):
        with Image.open(path) as im:
            if im.mode not in ("RGB", "L"):
                im = im.convert("RGB")
            elif im.mode == "L":
                im = im.convert("RGB")
            arr = np.array(im)
        return arr

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row["path"])
        label = np.array([row["label"]], dtype=np.float32)
        if self.transforms:
            aug = self.transforms(image=img)
            img = aug["image"] 
        return img, torch.from_numpy(label)


In [ ]:
train_ds = AerialWasteBinaryDataset(train_df, transforms=train_tfms)
val_ds   = AerialWasteBinaryDataset(val_df,   transforms=val_tfms)

#Class-Weighted Sampling
class_counts = train_df["label"].value_counts().to_dict()
weights = [1.0 / class_counts[row["label"]] for _, row in train_df.iterrows()]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,          
    pin_memory=PIN_MEMORY,            
    persistent_workers=False          
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,          
    pin_memory=PIN_MEMORY,            
    persistent_workers=False
)


In [ ]:
class ResNet50FPNBinary(nn.Module):
    def __init__(self, pretrained=True, fpn_out_channels=256, dropout=0.2):
        super().__init__()
        # 1) ResNet50 Backbone
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        self.backbone.fc = nn.Identity()  # wir nutzen nur Feature-Maps

        # Feature maps: layer1(C2)=256, layer2(C3)=512, layer3(C4)=1024, layer4(C5)=2048
        self.feat_extractor = create_feature_extractor(
            self.backbone,
            return_nodes={
                "layer1": "c2",
                "layer2": "c3",
                "layer3": "c4",
                "layer4": "c5",
            }
        )

        in_channels_list = [256, 512, 1024, 2048]
        self.fpn = FeaturePyramidNetwork(in_channels_list=in_channels_list, out_channels=fpn_out_channels)

        self.dropout = nn.Dropout(p=dropout)
        self.head = nn.Linear(4 * fpn_out_channels, 1)

    def forward(self, x):
        feats = self.feat_extractor(x)          # dict: {"c2":..., "c3":..., "c4":..., "c5":...}
        fpn_out = self.fpn(feats)         

        pooled = []
        for key in ["c5", "c4", "c3", "c2"]:
            p = fpn_out[key]
            gap = F.adaptive_avg_pool2d(p, output_size=1).flatten(1)  # (B, 256)
            pooled.append(gap)

        z = torch.cat(pooled, dim=1)            # (B, 1024)
        z = self.dropout(z)
        logit = self.head(z)                    # (B, 1)
        return logit


In [ ]:
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    return {"acc": acc, "f1": f1, "auc": auc, "thr": thr}

def run_epoch(model, loader, optimizer=None, scaler=None, train=True):
    model.train(train)
    y_true, y_prob = [], []
    running_loss = 0.0

    for imgs, labels in tqdm(loader, disable=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type="cuda" if DEVICE=="cuda" else "cpu", dtype=torch.float16, enabled=True):
                logits = model(imgs)             
                loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)

            if train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        y_prob.extend(probs.tolist())
        y_true.extend(labels.detach().cpu().numpy().ravel().tolist())

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(y_true), np.array(y_prob))
    return epoch_loss, metrics, (np.array(y_true), np.array(y_prob))


In [ ]:
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

def amp_autocast():
    if DEVICE == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    else:
        return contextlib.nullcontext()

scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

def mps_empty_cache():
    if DEVICE == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

# Hyperparameters
LR = globals().get("LR", 3e-4)
WEIGHT_DECAY = globals().get("WEIGHT_DECAY", 1e-4)
EPOCHS = globals().get("EPOCHS", 10)
CHECKPOINT = globals().get("CHECKPOINT", "resnet50_fpn_binary.pth")
GRAD_ACCUM_STEPS = globals().get("GRAD_ACCUM_STEPS", 2)   # >1 spart VRAM
FREEZE_BACKBONE_EPOCHS = globals().get("FREEZE_BACKBONE_EPOCHS", 3)  # erste Epochen einfrieren

# Utils
if "compute_metrics" not in globals():
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
    def compute_metrics(y_true, y_prob, thr=0.5):
        y_pred = (y_prob >= thr).astype(int)
        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_prob)
        except Exception:
            auc = float("nan")
        return {"acc": acc, "f1": f1, "auc": auc, "thr": thr}

# pos_weight saving
if "pos_weight" not in globals() or pos_weight is None:
    pos_weight = torch.tensor([1.0], dtype=torch.float32)

def set_backbone_requires_grad(model, requires_grad: bool):
    """
    Erwartet dein ResNet50+FPN Modell mit Attributen .backbone oder .feat_extractor.
    Passt für die vorgestellte ResNet50FPNBinary-Implementierung (torchvision).
    """
    backbone = getattr(model, "backbone", None)
    if backbone is not None:
        for p in backbone.parameters():
            p.requires_grad = requires_grad
    feat_extractor = getattr(model, "feat_extractor", None)
    if feat_extractor is not None:
        for p in feat_extractor.parameters():
            p.requires_grad = requires_grad

# Model
model = ResNet50FPNBinary(pretrained=True).to(DEVICE)

# Freezing backbome 
set_backbone_requires_grad(model, False)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

best_auc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{EPOCHS} (device={DEVICE}) ===")
    mps_empty_cache()

    if epoch == FREEZE_BACKBONE_EPOCHS + 1:
        set_backbone_requires_grad(model, True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        print("-> Backbone unfreezed.")

# Train
    model.train(True)
    y_true_tr, y_prob_tr = [], []
    running_loss = 0.0

    optimizer.zero_grad(set_to_none=True)
    for step, (imgs, labels) in enumerate(tqdm(train_loader, desc="Train", leave=False), start=1):
        imgs   = imgs.to(DEVICE, non_blocking=False)
        labels = labels.to(DEVICE, non_blocking=False).float()
        pw = pos_weight.to(imgs.device, dtype=torch.float32)

        with amp_autocast():
            logits = model(imgs)
            loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pw)
            loss = loss / GRAD_ACCUM_STEPS

        if scaler is not None:  
            scaler.scale(loss).backward()
        else:                   
            loss.backward()

        if step % GRAD_ACCUM_STEPS == 0:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            mps_empty_cache()

        running_loss += loss.item() * imgs.size(0) * GRAD_ACCUM_STEPS  
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        y_prob_tr.extend(probs.tolist())
        y_true_tr.extend(labels.detach().cpu().numpy().ravel().tolist())

    train_loss = running_loss / max(len(train_loader.dataset), 1)
    train_metrics = compute_metrics(np.array(y_true_tr), np.array(y_prob_tr))

# Validation
    model.eval()
    y_true_v, y_prob_v = [], []
    v_loss_sum, n_v = 0.0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Val", leave=False):
            imgs   = imgs.to(DEVICE, non_blocking=False)
            labels = labels.to(DEVICE, non_blocking=False).float()
            pw = pos_weight.to(imgs.device, dtype=torch.float32)
            with amp_autocast():
                logits = model(imgs)
                v_loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pw)
            v_loss_sum += v_loss.item() * imgs.size(0)
            n_v += imgs.size(0)
            y_prob_v.extend(torch.sigmoid(logits).cpu().numpy().ravel().tolist())
            y_true_v.extend(labels.cpu().numpy().ravel().tolist())
    val_loss = v_loss_sum / max(n_v, 1)
    val_metrics = compute_metrics(np.array(y_true_v), np.array(y_prob_v))

    print(
        f"Train | loss={train_loss:.4f} acc={train_metrics['acc']:.4f} "
        f"f1={train_metrics['f1']:.4f} auc={train_metrics['auc']:.4f}"
    )
    print(
        f"Val   | loss={val_loss:.4f} acc={val_metrics['acc']:.4f} "
        f"f1={val_metrics['f1']:.4f} auc={val_metrics['auc']:.4f}"
    )

    if (not math.isnan(val_metrics["auc"])) and (val_metrics["auc"] > best_auc):
        best_auc = val_metrics["auc"]
        best_state = {
            k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v)
            for k, v in model.state_dict().items()
        }
        torch.save(best_state, CHECKPOINT)
        print(f"Saved BEST checkpoint → {CHECKPOINT} (AUC={best_auc:.4f})")
        mps_empty_cache()

# Downloading best state
if best_state is not None:
    model.load_state_dict(best_state, strict=True)
    model.to(DEVICE).eval()
    mps_empty_cache()


=== Epoch 1/10 (device=mps) ===


Train | loss=0.5726 acc=0.5882 f1=0.7200 auc=0.5667
Val   | loss=3.5798 acc=0.3333 f1=0.5000 auc=0.0000
Saved BEST checkpoint → resnet50_fpn_binary.pth (AUC=0.0000)

=== Epoch 2/10 (device=mps) ===


Train | loss=1.8695 acc=0.5882 f1=0.7407 auc=0.8636
Val   | loss=0.7865 acc=0.6667 f1=0.0000 auc=0.0000

=== Epoch 3/10 (device=mps) ===


Train | loss=0.5808 acc=0.5294 f1=0.2000 auc=0.9861
Val   | loss=0.7207 acc=0.6667 f1=0.0000 auc=0.0000

=== Epoch 4/10 (device=mps) ===
-> Backbone unfreezed.


Train | loss=0.3826 acc=0.7647 f1=0.7500 auc=0.9857
Val   | loss=6.4430 acc=0.3333 f1=0.5000 auc=0.5000
Saved BEST checkpoint → resnet50_fpn_binary.pth (AUC=0.5000)

=== Epoch 5/10 (device=mps) ===


...

Train | loss=2.2755 acc=0.3529 f1=0.5217 auc=0.8429
Val   | loss=0.8612 acc=0.3333 f1=0.0000 auc=0.0000

=== Epoch 7/10 (device=mps) ===


Train | loss=0.1772 acc=0.8824 f1=0.8750 auc=0.9861
Val   | loss=1.2490 acc=0.6667 f1=0.0000 auc=0.0000

=== Epoch 8/10 (device=mps) ===


Train | loss=0.7740 acc=0.5882 f1=0.2222 auc=0.9861
Val   | loss=1.4127 acc=0.6667 f1=0.0000 auc=0.5000

=== Epoch 9/10 (device=mps) ===


Train | loss=0.5391 acc=0.7647 f1=0.5000 auc=1.0000
Val   | loss=0.8709 acc=0.3333 f1=0.0000 auc=0.5000

=== Epoch 10/10 (device=mps) ===


Train | loss=1.6463 acc=0.5294 f1=0.5000 auc=1.0000
Val   | loss=1.4183 acc=0.6667 f1=0.6667 auc=0.5000


In [ ]:
best_model = ResNet50FPNBinary(pretrained=False).to(DEVICE)
best_model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE), strict=True)
best_model.eval()

ResNet50FPNBinary(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(

In [ ]:
# Parse testing.json
with open(TEST_JSON, "r") as f:
    te = json.load(f)

test_items = []
for d in te.get("images", te):
    fn = d.get("file_name")
    p = find_image_path(fn)
    if p is not None:
        test_items.append({
            "file_name": fn,
            "path": str(p),
            "width": d.get("width"),
            "height": d.get("height"),
        })

test_df = pd.DataFrame(test_items)
print("Test Samples:", len(test_df))

class TestDataset(Dataset):
    def __init__(self, df, transforms):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["path"]) as im:
            if im.mode not in ("RGB", "L"):
                im = im.convert("RGB")
            elif im.mode == "L":
                im = im.convert("RGB")
            img = np.array(im)
        aug = self.transforms(image=img)
        return aug["image"], row["file_name"]

test_ds = TestDataset(test_df, transforms=val_tfms)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

pred_rows = []
with torch.no_grad():
    for imgs, fns in tqdm(test_loader, desc="Inference"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        logits = best_model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        for fn, p in zip(fns, probs):
            pred_rows.append({"file_name": fn, "waste_proba": float(p), "waste_pred": int(p >= 0.5)})

pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(PRED_CSV, index=False)
pred_df.head()


Test Samples: 2607


Inference:   0%|          | 0/326 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Inference: 100%|██████████| 326/326 [01:58<00:00,  2.76it/s]


,file_name,waste_proba,waste_pred
0,13.png,0.999982,1
1,15.png,0.999963,1
2,16.png,0.999970,1
3,17.png,0.999976,1
4,21.png,0.999860,1


In [ ]:
# Evaluate on validation and find a better threshold if needed
best_model.eval()
y_true, y_prob = [], []
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc="Val Eval"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = best_model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        y_prob.extend(probs.tolist())
        y_true.extend(labels.cpu().numpy().ravel().tolist())

y_true = np.array(y_true); y_prob = np.array(y_prob)

# grid search over thresholds
ths = np.linspace(0.1, 0.9, 17)
best = None
for t in ths:
    m = compute_metrics(y_true, y_prob, thr=t)
    if best is None or m["f1"] > best["f1"]:
        best = m
print("Best threshold on VAL:", best)

# Confusion Matrix bei best['thr']
y_pred = (y_prob >= best["thr"]).astype(int)
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix @thr=", best["thr"], "\n", cm)


Val Eval: 100%|██████████| 1/1 [00:00<00:00,  5.83it/s]

Best threshold on VAL: {'acc': 0.3333333333333333, 'f1': np.float64(0.5), 'auc': np.float64(0.5), 'thr': np.float64(0.1)}
Confusion Matrix @thr= 0.1 
 [[0 2]
 [0 1]]
